# 01 – Dataset: files → split → patches → DataLoaders

What this notebook does:
1. Checks **where it is running** (macOS for preparation, Windows + GPU for training).
2. Finds **all scan folders** in `Dataset/CTA_scans/` (`internal_Tr01`, `internal_Tr02`, … are picked up automatically) and the masks in `Dataset/Annotations/internal_annotation/`.
3. Makes a **train / val / test split by patient**, saved to `work/splits/splits.csv`. The split is stable: adding new scan folders later only adds new cases, and old cases keep their split.
4. Cuts one **64³ patch around every aneurysm** (0.45 mm voxels) and caches it in `work/patches/` (done once, resumable).
5. Builds **PyTorch Datasets and DataLoaders** and checks one batch by eye.

All paths saved to CSV are relative to the project folder, so `work/` can be copied from the Mac to the Windows machine, or this notebook can be re-run there.

In [ ]:
# ---- Setup: make the code in src/ importable ----
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()          # this notebook lives in the project root
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aneurysm.config import load_data_config, find_project_root
from aneurysm.env import describe_environment, os_name, default_num_workers
from aneurysm.data import (build_catalog, load_or_create_splits,
                           build_patch_index, load_patch_index, make_dataloaders)
from aneurysm.data.catalog import summarize_catalog

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

## 1. Where are we running?

In [ ]:
env = describe_environment()
for k, v in env.items():
    print(f"{k:>15}: {v}")

RUNNING_ON = os_name()
if RUNNING_ON == "macos":
    print("\n-> macOS: preparing data. Training will run on the Windows GPU machine.")
elif RUNNING_ON == "windows":
    print("\n-> Windows:", "GPU found." if env.get("cuda_available") else
          "NO GPU found - check the CUDA build of PyTorch (see requirements.txt).")

## 2. Settings
All dataset settings are in **`configs/data.yaml`**. Edit that file, then re-run from here.

In [ ]:
cfg = load_data_config(PROJECT_ROOT / "configs" / "data.yaml", project_root=PROJECT_ROOT)

print('Model patch (x, y, z):', cfg.model_patch_shape, '| stored on disk:', cfg.stored_patch_shape)
pd.Series(cfg.to_dict())

## 3. Find all scans and masks

In [ ]:
catalog = build_catalog(cfg)
print("Scan folders found:", sorted(catalog.loc[catalog.has_image, "scan_folder"].unique()))
display(summarize_catalog(catalog))
print("Paired cases (scan + mask):", int(catalog.paired.sum()))

orphans = catalog[catalog.has_image & ~catalog.has_mask]
if len(orphans):
    print("Scans without a mask:", orphans.case_id.tolist())

## 4. Train / val / test split (by patient)

In [ ]:
splits = load_or_create_splits(catalog, cfg)
print("Saved to:", cfg.rel(cfg.splits_path))
print("New cases added this run:", int(splits.is_new.sum()))

display(pd.crosstab(splits.split, splits.official_subset, margins=True))
display(pd.crosstab(splits.institution_id, splits.split, margins=True))

With only `internal_Tr01` (99 cases) the split is roughly 70 / 15 / 15. Official `Ts` cases go straight to **test** once their scans are downloaded.

## 5. Cut and cache aneurysm patches
Each aneurysm becomes one stored patch (model patch + an 8-voxel margin on every side, for random shifts), at 0.45 mm.

Each patch setting is cached in its own folder, e.g. `work/patches/64x64x32_sp0.45_hu-100to900_m8_min20/`. Changing `data.yaml` (for example to 64x64x64) simply creates a new folder; switching back reuses the old one. No manual overwrite needed.
First run on 99 scans takes a few minutes. Later runs only process new cases.

In [ ]:
OVERWRITE = False   # only to force re-cutting the SAME setting (e.g. after a code change)

index = build_patch_index(catalog, splits, cfg, overwrite=OVERWRITE, n_threads=2)
print("Patch index:", cfg.rel(cfg.patch_index_path))
display(index.groupby("split").agg(patches=("patch_id", "size"),
                                   cases=("case_id", "nunique"),
                                   median_diam_mm=("diameter_mm", "median")))

In [ ]:
# Size of the aneurysms we will train on
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for s, g in index.groupby("split"):
    ax[0].hist(g.diameter_mm, bins=np.arange(0, 25, 1), alpha=0.6, label=f"{s} (n={len(g)})")
ax[0].set(xlabel="aneurysm diameter (mm)", ylabel="patches", title="Aneurysm size per split"); ax[0].legend()
side_mm = min(cfg.model_patch_shape) * cfg.spacing_mm     # smallest side (depth for 32 slices)
too_big = index[index.diameter_mm > side_mm * 0.8]
ax[1].axis("off")
ax[1].text(0, 0.6, f"Patch: {cfg.model_patch_shape} voxels, smallest side {side_mm:.1f} mm\n"
                   f"Aneurysms > 80% of patch side: {len(too_big)}\n"
                   f"(they will be cut at the patch edge)", fontsize=11)
plt.tight_layout(); plt.show()

## 6. Datasets and DataLoaders

In [ ]:
BATCH_SIZE = 2           # small on purpose: the pilot GPU has 4 GB
loaders = make_dataloaders(index, cfg, batch_size=BATCH_SIZE)   # workers: 0 on Mac, 2 on Windows

for split, dl in loaders.items():
    print(f"{split:>5}: {len(dl.dataset):4d} patches, {len(dl):4d} batches, workers={dl.num_workers}")

batch = next(iter(loaders["train"]))
for k, v in batch.items():
    print(f"{k:>9}:", tuple(v.shape) if hasattr(v, "shape") else v,
          (f"dtype={v.dtype}, min={v.min():.2f}, max={v.max():.2f}" if hasattr(v, "dtype") else ""))

## 7. Look at a few patches
Middle axial slice through the centred aneurysm. **Yellow** = the centred aneurysm (used for the histogram), **red** = all aneurysms in the patch (used for the loss).

In [ ]:
def show_items(ds, n=4, seed=0):
    rng = np.random.default_rng(seed)
    ids = rng.choice(len(ds), size=min(n, len(ds)), replace=False)
    fig, axs = plt.subplots(2, len(ids), figsize=(3.2 * len(ids), 6.2), squeeze=False)
    for j, i in enumerate(ids):
        item = ds[int(i)]
        img, msk = item["image"][0].numpy(), item["mask"][0].numpy()     # (D, H, W)
        ctr = item["center_mask"][0].numpy()
        z = int(np.argmax(ctr.sum(axis=(1, 2)))) if ctr.any() else img.shape[0] // 2
        axs[0, j].imshow(img[z], cmap="gray", vmin=-1, vmax=1, origin="lower")
        axs[0, j].contour(msk[z], levels=[0.5], colors="r", linewidths=1, origin="lower")        # all aneurysms
        if ctr[z].any():
            axs[0, j].contour(ctr[z], levels=[0.5], colors="yellow", linewidths=1, origin="lower")  # centred one
        axs[0, j].set_title(f"{item['patch_id']}  (slice {z})", fontsize=9); axs[0, j].axis("off")
        axs[1, j].bar(np.arange(len(item["hist"])), item["hist"].numpy())
        axs[1, j].set_title("histogram (centred aneurysm)", fontsize=9)
    plt.tight_layout(); plt.show()

for split in loaders:
    print(split); show_items(loaders[split].dataset)

## 8. Checks before moving on
- **Split**: every case is in exactly one split; all patches of a case share its split (checked below).
- **Masks**: every patch contains aneurysm voxels.
- **Moving to Windows**: copy the whole project folder (including `work/`), or simply re-run this notebook there. Paths in the CSVs are relative, so both work.

In [ ]:
assert splits.case_id.is_unique, "a case appears twice in the split"
per_case = index.groupby("case_id").split.nunique()
assert (per_case == 1).all(), "patches of one case are in different splits"
print("Patches with an empty mask:", int((index.mask_voxels_in_patch == 0).sum()))
print("Patches whose centred aneurysm is missing:", int((index.center_voxels_in_patch == 0).sum()))
print("Patches that also contain another aneurysm:", int((index.mask_voxels_in_patch > index.center_voxels_in_patch).sum()))
print("All checks passed.")